# Convolutional Neural Network

#### Note: download tensorflow and pillow first

### Importing the libraries

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress INFO and WARNING
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow import keras
!pip install scipy


2025-01-27 14:33:54.228818: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738006434.237028   18787 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738006434.239553   18787 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


  Using cached scipy-1.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached scipy-1.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (40.2 MB)


In [ ]:
tf.__version__

'2.18.0'

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [ ]:
# to avoid overfitting
train_datagen = ImageDataGenerator(rescale = 1./255,  #feature scaling to all pixels to [0,1]
                                   shear_range = 0.2, #transformations
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('dataset/training_set', #path
                                                 target_size = (64, 64), #training size
                                                 batch_size = 32,        #images taking each time
                                                 class_mode = 'binary')

Found 8000 images belonging to 2 classes.


### Preprocessing the Test set

In [ ]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


## Part 2 - Building the CNN

### Initialising the CNN

In [ ]:
cnn = keras.models.Sequential()

# Here I added the input layer explicitly
cnn.add(keras.layers.Input(shape=(64, 64, 3)))

In [ ]:
#colored image in RGB, it's 64x64 as defined above, so it's 64x64x3
cnn.add(keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))


I0000 00:00:1738006449.951376   18787 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 989 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


### Step 2 - Pooling

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2)) #pool_size is the dimension of pooling matrix

### Adding a second convolutional layer

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu')) # input shape is deleted
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [ ]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [ ]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu')) #number of hidden neurons

### Step 5 - Output Layer

In [ ]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid')) #binary output for cats or dogs

## Part 3 - Training the CNN

### Compiling the CNN

In [ ]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [ ]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

Epoch 1/25


/home/locutus/tf_env/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1738006455.586573   18973 service.cc:148] XLA service 0x71751400a930 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1738006455.586589   18973 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
I0000 00:00:1738006455.667004   18973 cuda_dnn.cc:529] Loaded cuDNN version 90300


  8/250 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.4986 - loss: 0.7453

I0000 00:00:1738006456.230944   18973 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5532 - loss: 0.6875

/home/locutus/tf_env/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


250/250 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.5534 - loss: 0.6874 - val_accuracy: 0.6595 - val_loss: 0.6157
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.6615 - loss: 0.6142 - val_accuracy: 0.6865 - val_loss: 0.5900
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - accuracy: 0.7126 - loss: 0.5658 - val_accuracy: 0.6415 - val_loss: 0.6496
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.7283 - loss: 0.5427 - val_accuracy: 0.7620 - val_loss: 0.5045
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.7620 - loss: 0.4982 - val_accuracy: 0.7805 - val_loss: 0.4754
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.7639 - loss: 0.4873 - val_accuracy: 0.7910 - val_loss: 0.4773
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.7821 - loss: 0.4594 - val_accuracy: 0.7640 - val_loss: 0.4884
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.7811 - loss: 0.4536 - val_accuracy: 0.76

## Part 4 - Making a single prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step


In [ ]:
print(prediction)

dog


# Here I make a single prediction an another image in the set Answer for 2 (A).

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_2.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


In [ ]:
print(prediction)

cat


# Here is my answer for 2 (C).

* 1. First I load the dircotries and get the number of Test Images.

In [ ]:
import os

# Here I define paths to the cats and dogs folders
test_set_path = 'dataset/test_set'
cats_folder = os.path.join(test_set_path, 'cats')
dogs_folder = os.path.join(test_set_path, 'dogs')

# Next I get the list of images in the cats and dogs folders
cats_images = os.listdir(cats_folder)
dogs_images = os.listdir(dogs_folder)

# Then I define the total number of test images (assuming same number of cats and dogs)
Ntest = len(cats_images)
print(Ntest)

1001


* 2. Second I randomly seclected 10 indices.

In [ ]:
import random

# Here I randomly draw 10 indices from 0 to Ntest - 1
random_indices = random.sample(range(Ntest), 10)

* 3. Third I initlize my list of predictions.

In [ ]:
# Here I initalize the list for storing predictions and true labels
predictions = []
true_labels = []

* 4. Fourth, I make my predictions for cat images.

In [ ]:
import time

# Here I am inializing my lists for predictions, true labels, and image paths
predictions = []
true_labels = []
image_paths = []

# For cat images
for index in random_indices:
    # Loading and preprocess the cat image
    cat_image_path = os.path.join(cats_folder, cats_images[index])
    test_image = image.load_img(cat_image_path, target_size=(64, 64))
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis=0)

    # Predicting the class
    result = cnn.predict(test_image)
    if result[0][0] == 1:
        prediction = 'dog'
    else:
        prediction = 'cat'

    # Append to lists
    predictions.append(prediction)
    true_labels.append('cat')
    image_paths.append(cat_image_path)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


* 5. Fith, I make predictions for the dog images.

In [ ]:
import time

# For dog images
for index in random_indices:
    # Loading and preprocess the dog image
    dog_image_path = os.path.join(dogs_folder, dogs_images[index])
    test_image = image.load_img(dog_image_path, target_size=(64, 64))
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis=0)

    # Predicting the class
    result = cnn.predict(test_image)
    if result[0][0] == 1:
        prediction = 'dog'
    else:
        prediction = 'cat'

    # Append to lists
    predictions.append(prediction)
    true_labels.append('dog')
    image_paths.append(dog_image_path)




1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step


*6. Finally I compare my predictions with the true labels and calculate my accuracy

In [ ]:
# Now I compare the predictions with true labels
correct_predictions = 0
for pred, true, image_path in zip(predictions, true_labels, image_paths):
    if pred == true:
        correct_predictions += 1
    # Optionally print individual image predictions and labels with path
    print(f"Image: {image_path} - Predicted: {pred} - True: {true}")

# Calculate accuracy
accuracy = correct_predictions / len(predictions) * 100

# Print results
print(f"Total correct predictions: {correct_predictions} out of {len(predictions)}")
print(f"Accuracy: {accuracy:.2f}%")




Image: dataset/test_set/cats/cat.4448.jpg - Predicted: dog - True: cat
Image: dataset/test_set/cats/cat.4426.jpg - Predicted: dog - True: cat
Image: dataset/test_set/cats/cat.4650.jpg - Predicted: dog - True: cat
Image: dataset/test_set/cats/cat.4678.jpg - Predicted: dog - True: cat
Image: dataset/test_set/cats/cat.4664.jpg - Predicted: dog - True: cat
Image: dataset/test_set/cats/cat.4112.jpg - Predicted: dog - True: cat
Image: dataset/test_set/cats/cat.4406.jpg - Predicted: dog - True: cat
Image: dataset/test_set/cats/cat.4171.jpg - Predicted: dog - True: cat
Image: dataset/test_set/cats/cat.4588.jpg - Predicted: cat - True: cat
Image: dataset/test_set/cats/cat.4122.jpg - Predicted: cat - True: cat
Image: dataset/test_set/dogs/dog.4077.jpg - Predicted: dog - True: dog
Image: dataset/test_set/dogs/dog.4738.jpg - Predicted: dog - True: dog
Image: dataset/test_set/dogs/dog.4741.jpg - Predicted: dog - True: dog
Image: dataset/test_set/dogs/dog.4711.jpg - Predicted: dog - True: dog
Image: